In [1]:
%load_ext autoreload
%autoreload 2

# 1. DQN Agent

In [2]:
import numpy as np
import torch

from briscola import Briscola
from agents.dqn_agent import DQN_Agent, state_to_tensor

In [3]:
N_EPISODES = 100_000
LOG_INTERVAL = 500

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [4]:
env = Briscola()
agent = DQN_Agent(env, device)

In [5]:
episode_rewards = [] # Hopefully higher as episodes increase
wins = 0
initial_epsilon = agent.epsilon

for i in range(N_EPISODES):
    done = False
    ep_reward = 0.0
    
    state, _ = env.reset()
    
    while not done:
        # s_tensor = state_to_tensor(s, env_dqn).to(agent.device)
        action = agent.select_action(state).item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        next_mask = next_state["hand"]
        done = terminated or truncated

        ep_reward += reward
        
        # Convert to tensors to store in experience buffer
        s_tensor = state_to_tensor(state).to(agent.device)
        r_tensor = torch.tensor([reward], dtype = torch.float32, device = agent.device)
        a_tensor = torch.tensor([[action]], device = agent.device, dtype = torch.long)

        if terminated:
            ns_tensor = None
            nm_tensor = None
        else:
            ns_tensor = state_to_tensor(next_state).to(agent.device)
            nm_tensor = torch.tensor(next_mask, dtype = torch.float32, device = agent.device).unsqueeze(0)
        
        # Buffer update
        agent.buffer.push(s_tensor, a_tensor, ns_tensor, r_tensor, nm_tensor)
        
        state = next_state
        action_mask = next_mask
        
        # Execute a learning step
        agent.learn()

    agent.epsilon = max(agent.eps_end, initial_epsilon - (i / N_EPISODES) * (initial_epsilon - agent.eps_end))
    
    episode_rewards.append(ep_reward)
    if env.player_score > 60:
        wins += 1

    if (i + 1) % LOG_INTERVAL == 0:
        recent      = episode_rewards[-LOG_INTERVAL:]
        win_rate    = wins / (i + 1) * 100
        print(
            f"Episode {i+1:>6}/{N_EPISODES} | "
            f"Mean reward (last {LOG_INTERVAL}): {np.mean(recent):+.2f} | "
            f"Win rate: {win_rate:.1f}% | "
            f"Epsilon: {agent.epsilon:.4f}"
        )

torch.save(agent.policy_net.state_dict(), "briscola_dqn.pt")
print("Saved.")

Episode    500/100000 | Mean reward (last 500): -17.82 | Win rate: 45.8% | Epsilon: 0.8956
Episode   1000/100000 | Mean reward (last 500): -2.61 | Win rate: 47.3% | Epsilon: 0.8911
Episode   1500/100000 | Mean reward (last 500): -3.03 | Win rate: 47.8% | Epsilon: 0.8867
Episode   2000/100000 | Mean reward (last 500): +10.25 | Win rate: 48.8% | Epsilon: 0.8822
Episode   2500/100000 | Mean reward (last 500): +6.73 | Win rate: 49.1% | Epsilon: 0.8778
Episode   3000/100000 | Mean reward (last 500): +8.01 | Win rate: 49.4% | Epsilon: 0.8733
Episode   3500/100000 | Mean reward (last 500): -10.34 | Win rate: 49.0% | Epsilon: 0.8689
Episode   4000/100000 | Mean reward (last 500): +4.54 | Win rate: 49.1% | Epsilon: 0.8644
Episode   4500/100000 | Mean reward (last 500): +10.05 | Win rate: 49.4% | Epsilon: 0.8600
Episode   5000/100000 | Mean reward (last 500): +22.57 | Win rate: 49.9% | Epsilon: 0.8555
Episode   5500/100000 | Mean reward (last 500): +18.61 | Win rate: 50.2% | Epsilon: 0.8511
Epis

# 1. First implementation

In [ ]:
# set up matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

In [ ]:
def plot_rewards(show_result=False):
    plt.figure(1)
    rewards_t = torch.tensor(episode_rewards, dtype=torch.float)

    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')

    plt.xlabel('Episode')
    plt.ylabel('Total Reward')

    # Plot raw rewards
    # plt.plot(rewards_t.numpy(), label="Episode Reward")

    # Plot moving average (100 episodes)
    # if len(rewards_t) >= 100:
    if len(rewards_t) >= 10:
        # means = rewards_t.unfold(0, 100, 1).mean(1).view(-1)
        means = rewards_t.unfold(0, 10, 1).mean(1).view(-1)
        # means = torch.cat((torch.zeros(99), means))
        means = torch.cat((torch.zeros(9), means))
        plt.plot(means.numpy(), label="10-episode avg")

    plt.legend()
    # plt.pause(0.001)

    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

In [ ]:
num_episodes = 100_000
win_rates = []

for i_episode in range(num_episodes):
    # Initialize the environment and get its state
    state, info = env.reset()
    state = flatten_obs(state)
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    total_reward = 0
    done = False
    while not done:
        action_mask = info["action_masks"]
        action = select_action(state, action_mask)
        observation, reward, terminated, truncated, info = env.step(action.item())
        observation = flatten_obs(observation)
        reward = torch.tensor([reward], device=device)
        done = terminated or truncated

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # Store the transition in memory
        memory.push(state, action, next_state, reward)

        # Move to the next state
        state = next_state

        # Perform one step of the optimization (on the policy network)
        optimize_model()

        # Soft update of the target network's weights
        # θ′ ← τ θ + (1 −τ )θ′
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
        target_net.load_state_dict(target_net_state_dict)

        total_reward += reward.item()  # Convert to scalar
    episode_rewards.append(total_reward)
    
    # Track win rate every 100 episodes
    if (i_episode + 1) % 100 == 0:
        wins = sum(1 for r in episode_rewards[-100:] if r > 0)  # Assuming positive reward means win
        win_rate = wins / 100
        win_rates.append(win_rate)
        print(f"Episode {i_episode+1}: Win rate last 100: {win_rate:.3f}")
    
    plot_rewards()

print('Complete')
plot_rewards(show_result=True)
plt.ioff()
plt.show()

In [ ]:
# Evaluate the trained agent
def evaluate_agent(env, agent, n_games=1000):
    wins = 0
    total_rewards = []
    
    for _ in range(n_games):
        obs, info = env.reset()
        obs = flatten_obs(obs)
        obs = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        done = False
        episode_reward = 0
        
        while not done:
            mask = info["action_masks"]
            with torch.no_grad():
                q_values = policy_net(obs)
                mask_tensor = torch.tensor(mask, device=device).unsqueeze(0)
                q_values[mask_tensor == 0] = -1e9
                action = q_values.max(1).indices.view(1, 1)
            
            obs_next, reward, terminated, truncated, info = env.step(action.item())
            obs_next = flatten_obs(obs_next)
            obs_next = torch.tensor(obs_next, dtype=torch.float32, device=device).unsqueeze(0)
            obs = obs_next
            episode_reward += reward
            done = terminated or truncated
        
        if env.player_score > env.opponent_score:
            wins += 1
        total_rewards.append(episode_reward)
    
    win_rate = wins / n_games
    mean_reward = np.mean(total_rewards)
    print(f"Win rate: {win_rate:.3f}")
    print(f"Mean reward: {mean_reward:.3f}")
    return win_rate, mean_reward

win_rate, mean_reward = evaluate_agent(env, policy_net, 1000)

In [ ]:
# Inspect Q-values for a sample state
def inspect_q_values(env, policy_net, num_samples=5):
    for i in range(num_samples):
        obs, info = env.reset()
        obs_flat = flatten_obs(obs)
        obs_tensor = torch.tensor(obs_flat, dtype=torch.float32, device=device).unsqueeze(0)
        
        with torch.no_grad():
            q_values = policy_net(obs_tensor).squeeze(0).cpu().numpy()
        
        mask = info["action_masks"]
        valid_indices = np.where(mask == 1)[0]
        
        print(f"Sample {i+1}:")
        print(f"  Hand: {[f'{idx//10}-{idx%10+1}' for idx in valid_indices]}")
        print(f"  Briscola suit: {np.argmax(obs['briscola'])}")
        print(f"  Is first: {obs['is_first'][0]}")
        print(f"  Table: {obs['table_card']}")
        print(f"  Q-values for valid actions: {q_values[valid_indices]}")
        print(f"  Best action: {valid_indices[np.argmax(q_values[valid_indices])]} (Q={np.max(q_values[valid_indices]):.3f})")
        print()

inspect_q_values(env, policy_net, 3)

___

# 2. PPO Agent

In [ ]:
import numpy as np
import torch
from briscola import Briscola
from agents.ppo_agent import BriscolaNet, PPO, RolloutBuffer

# ── Hyperparameters ──────────────────────────────────────────
ROLLOUT_STEPS  = 2048   # steps to collect before each update
TOTAL_STEPS    = 1_000_000
LR             = 3e-4
GAMMA          = 0.99
GAE_LAMBDA     = 0.95
CLIP_EPS       = 0.2
VALUE_COEF     = 0.5
ENTROPY_COEF   = 0.03   # increase to 0.05 if agent gets stuck early
N_EPOCHS       = 10
BATCH_SIZE     = 64
MAX_GRAD_NORM  = 0.5
HIDDEN         = 128
LOG_INTERVAL   = 10     # log every N updates
# ─────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu")

def flatten_obs(obs: dict) -> np.ndarray:
    return np.concatenate([
        obs["hand"].astype(np.float32),
        obs["table_card"].astype(np.float32),
        obs["briscola"].astype(np.float32),
        obs["played_cards"].astype(np.float32),
        obs["is_first"].astype(np.float32),
    ])

env       = Briscola()
obs_dim   = len(flatten_obs(env.reset()[0]))
n_actions = env.action_space.n   # 40

net = BriscolaNet(obs_dim, n_actions, hidden=HIDDEN).to(device)
agent = PPO(
    net, lr=LR, gamma=GAMMA, gae_lambda=GAE_LAMBDA,
    clip_eps=CLIP_EPS, value_coef=VALUE_COEF,
    entropy_coef=ENTROPY_COEF, n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE, max_grad_norm=MAX_GRAD_NORM,
    device=str(device),
)

buffer = RolloutBuffer()

obs, info   = env.reset()
state       = flatten_obs(obs)
action_mask = info["action_masks"]

episode_rewards = []
ep_reward       = 0.0
total_steps     = 0
n_updates       = 0

while total_steps < TOTAL_STEPS:

    # ── Phase 1: collect ROLLOUT_STEPS transitions ──
    buffer.clear()
    for _ in range(ROLLOUT_STEPS):
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        mask_t  = torch.tensor(action_mask, dtype=torch.bool, device=device).unsqueeze(0)

        with torch.no_grad():
            action, log_prob, _, value = net.get_action(state_t, mask_t)

        a = action.item()
        next_obs, reward, terminated, truncated, info = env.step(a)
        done = terminated or truncated

        buffer.states.append(state)
        buffer.actions.append(a)
        buffer.log_probs.append(log_prob.item())
        buffer.rewards.append(reward)
        buffer.values.append(value.item())
        buffer.masks.append(action_mask)
        buffer.dones.append(float(done))

        ep_reward += reward
        total_steps += 1

        if done:
            episode_rewards.append(ep_reward)
            ep_reward = 0.0
            obs, info = env.reset()
        else:
            obs = next_obs

        state       = flatten_obs(obs)
        action_mask = info["action_masks"]

    # ── Phase 2 & 3: compute GAE + update ──
    last_state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    last_mask_t  = torch.tensor(action_mask, dtype=torch.bool, device=device).unsqueeze(0)
    with torch.no_grad():
        _, last_value = net(last_state_t)
    last_value = last_value.item()

    stats = agent.update(buffer, last_value)
    n_updates += 1

    if n_updates % LOG_INTERVAL == 0 and episode_rewards:
        recent = episode_rewards[-100:]
        print(
            f"Steps: {total_steps:>8,} | "
            f"Updates: {n_updates:>4} | "
            f"Mean reward (100 ep): {np.mean(recent):+.2f} | "
            f"Policy loss: {stats['policy_loss']:+.4f} | "
            f"Value loss: {stats['value_loss']:.4f} | "
            f"Entropy: {stats['entropy']:.4f}"
        )

torch.save(net.state_dict(), "briscola_ppo.pt")
print("Saved.")